# 第 2 周末练习解答 —— 技术问答 Gradio 原型

## 练习目标（理念）

把第 1 周的技术问答器升级成**完整原型**，用上第 2 周学过的能力：

- **Gradio UI**：聊天界面，不用手写前端
- **流式输出（streaming）**：边生成边显示，用 `yield` 推送增量
- **System Prompt**：用 system 消息定「资深软件工程师」人设与答法
- **模型切换**：下拉框在 GPT / Claude / Gemini 之间切换（经 OpenRouter）
- **加分项**：工具调用、语音输入输出（本解答聚焦前四项）

## 和本课第 2 周的关系

| 本课概念 | 本笔记本里你会看到 |
|----------|-------------------|
| Gradio `ChatInterface` | 最后一格启动 Web UI |
| `stream=True` + generator | `chat()` 里逐块 `yield` |
| System / User messages | `system_message` + history + 当前问题 |
| 多模型路由 | `MODEL_MAP` + OpenRouter `base_url` |

## 怎么跑

1. `.env` 里准备 `OPENROUTER_API_KEY`
2. 从上到下运行单元格
3. 最后一格会打开 Gradio；上方选模型，再提问


In [ ]:
# ========== 导入：把后面要用的工具箱搬进来 ==========

# 标准库 os：读环境变量（Environment Variables），例如 OPENROUTER_API_KEY
import os
# 标准库 json：本格导入备用（解析 JSON 时常用；本解答主路径未必用到）
import json
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# 从 openai 导入 OpenAI 客户端：这里用它的「OpenAI 兼容」接口去打 OpenRouter / 各厂商网关
from openai import OpenAI
# Gradio：快速搭聊天 Web UI（ChatInterface、Dropdown 等）
import gradio as gr


In [ ]:
# ========== 常量：模型 ID 与各厂商 base_url 集中管理 ==========

# OpenRouter 上的 GPT 路由名（字符串/model id 必须保持原样，改了就找不到模型）
MODEL_GPT = 'openai/gpt-4o-mini'
# OpenRouter 上的 Claude 路由名
MODEL_CLAUDE = 'anthropic/claude-sonnet-4.5'
# OpenRouter 上的 Gemini 路由名
MODEL_GEMINI = 'google/gemini-2.5-flash-lite'

# Anthropic 官方 OpenAI 兼容端点（URL 不翻译、不改）
anthropic_url = "https://api.anthropic.com/v1/"
# Google Gemini 的 OpenAI 兼容端点
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
# OpenRouter 统一网关：一个 key 可路由多家模型
openrouter_url = "https://openrouter.ai/api/v1"


In [ ]:
# ========== 初始化：读密钥 + 创建多个 OpenAI 兼容客户端 ==========

# 加载 .env；override=True 表示用文件里的值覆盖已有环境变量
load_dotenv(override=True)

# 从环境变量取出 OpenRouter API Key（密钥不要写进笔记本正文）
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
# 粗检：有 key 就打印前几个字符方便排查；打印文案保持英文原样
if openrouter_api_key:
    print(f"OpenAI API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenAI API Key not set")


# 下面三个客户端都是 OpenAI SDK，但 base_url 指向不同网关
# 注意：原代码把 openrouter_url 传给了 api_key 参数——逻辑保持原样，不在此「修正」
anthropic = OpenAI(api_key=openrouter_url, base_url=anthropic_url)
gemini = OpenAI(api_key=openrouter_url, base_url=gemini_url)
# 实际聊天走 openrouter：base_url + 真正的 OPENROUTER_API_KEY
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)


OpenAI API Key exists and begins sk-


In [ ]:
# ========== System Prompt：定人设与答法（发给模型的英文指令勿改译）==========

# system_message：会作为 role=system 的 content；改译会改变模型行为，故保留英文
system_message = """
You are a software engineer with 20 years of experience in the industry.
You are going to answer questions about the software development industry.
You will give concise answers which are not too long.
"""


In [ ]:
# ========== 模型下拉：界面显示名 → OpenRouter model id ==========

# 字典：Gradio 下拉框显示 "GPT"/"Claude"/"Gemini"，真正请求时用右边的 MODEL_* 常量
MODEL_MAP = {"GPT": MODEL_GPT, "Claude": MODEL_CLAUDE, "Gemini": MODEL_GEMINI}

# Gradio Dropdown：作为 ChatInterface 的 additional_inputs，把用户选择传给 chat()
model_selector = gr.Dropdown(["GPT", "Claude", "Gemini"], label="Select model", value="GPT")


In [ ]:
# ========== 核心 chat：拼 messages、按模型流式请求、逐块 yield ==========

def chat(message, history, model):
    # 把 Gradio history 规整成 [{role, content}, ...]（只取 role/content，丢掉多余字段）
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # messages = system + 历史轮次 + 当前用户问题
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 用下拉框选中的显示名查 MODEL_MAP；找不到则回退 MODEL_GPT
    model_id = MODEL_MAP.get(model, MODEL_GPT)
    # 经 OpenRouter 发起流式 Chat Completions（stream=True）
    stream = openrouter.chat.completions.create(model=model_id, messages=messages, stream=True)
    # 累加增量文本；每来一块就 yield 一次完整当前回复（Gradio 流式刷新用）
    response = ""
    for chunk in stream:
        # delta.content 可能为 None（例如纯 role 块），用 or '' 避免拼出 TypeError
        response += chunk.choices[0].delta.content or ''
        yield response


In [ ]:
# ========== 启动 Gradio：ChatInterface + 模型下拉 ==========

# ChatInterface：内置聊天气泡；fn=chat；type="messages" 表示 history 用 OpenAI 风格消息列表
view = gr.ChatInterface(
    fn=chat,
    type="messages",
    # 额外输入：上面定义的 model_selector，会作为 chat 的第三个参数 model
    additional_inputs=[model_selector],
    # UI 标题/说明字符串保持原样（展示给用户，不改逻辑）
    title="Technical Q&A Assistant",
    description="Ask technical questions about programming. Choose your model above.",
# launch()：起本地 Web 服务；返回值赋给 view（便于 notebook 里持有引用）
).launch()
